In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CARREGAMENTO DOS DADOS E BUSCA BLINDADA
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: Arquivo {file_path} não encontrado.")
    raise

# Mapeamento das Colunas de Interesse (Uso de drogas)
drogas_alvo = ['Crack', 'Álcool', 'Cannabis', 'Cocaína', 'Tabaco', 'Outras Drogas']
cols_drogas = [c for c in df.columns if 'choice=Sim' in c and any(d in c for d in drogas_alvo)]

# Encontrar a coluna de encaminhamento para a Psiquiatria de forma segura
col_psiquiatria_list = [c for c in df.columns if 'Para qual especialidade' in c and 'choice=Psiquiatria' in c]

if not col_psiquiatria_list:
    print("ERRO: Coluna de encaminhamento para Psiquiatria não encontrada.")
else:
    col_psiquiatria = col_psiquiatria_list[0]

    # ==============================================================================
    # 2. TRATAMENTO LONGITUDINAL (PROPAGAÇÃO)
    # ==============================================================================
    df = df.sort_values(['Record ID', 'Repeat Instance'], na_position='first')
    cols_to_fill = cols_drogas + [col_psiquiatria]
    df[cols_to_fill] = df.groupby('Record ID')[cols_to_fill].ffill()

    # Filtrar apenas as consultas reais
    df_ev = df[df['Repeat Instrument'].notna()].copy()

    # ==============================================================================
    # 3. CÁLCULO DAS MÉTRICAS (POLIUSO E PSIQUIATRIA)
    # ==============================================================================
    # Somamos quantas colunas de drogas estão marcadas como 'Checked'
    df_ev['Numero_Drogas'] = (df_ev[cols_drogas] == 'Checked').sum(axis=1)

    # Converter a coluna de psiquiatria para valor binário (1 = Sim, 0 = Não)
    df_ev['Encaminhado_Psiquiatria'] = (df_ev[col_psiquiatria] == 'Checked').astype(int)

    # ==============================================================================
    # 4. GERAÇÃO E EXPORTAÇÃO DAS BASES PARA O ARTIGO (NOVO!)
    # ==============================================================================
    
    # Base 1: Tabela de Agrupamento (A que gera o gráfico)
    tabela_export = df_ev.groupby('Numero_Drogas').agg(
        Total_Atendimentos=('Encaminhado_Psiquiatria', 'count'),
        Encaminhamentos_Psiquiatria=('Encaminhado_Psiquiatria', 'sum')
    ).reset_index()
    
    tabela_export['Taxa de Encaminhamento (%)'] = (tabela_export['Encaminhamentos_Psiquiatria'] / tabela_export['Total_Atendimentos'] * 100).round(2)
    
    tabela_export.columns = ['Índice de Poliuso (Nº Drogas)', 'Total de Atendimentos (N)', 'Encaminhamentos (N)', 'Taxa de Encaminhamento (%)']
    tabela_export.to_csv('base_poliuso_psiquiatria.csv', index=False, encoding='utf-8-sig')

    # ==============================================================================
    # 5. TESTE ESTATÍSTICO E EXPORTAÇÃO
    # ==============================================================================
    corr_coef, p_value = stats.pointbiserialr(df_ev['Encaminhado_Psiquiatria'], df_ev['Numero_Drogas'])
    
    # Base 2: Resultados do Teste Estatístico
    tabela_stats = pd.DataFrame({
        'Métrica': ['Correlação Ponto-Biserial (r)', 'Valor-p (Significância)', 'N Total Analisado'],
        'Valor': [round(corr_coef, 4), p_value, len(df_ev)]
    })
    tabela_stats.to_csv('base_estatistica_poliuso.csv', index=False, encoding='utf-8-sig')

    print(f"\n{'='*80}")
    print("✅ BASES DE DADOS EXPORTADAS COM SUCESSO:")
    print("- 'base_poliuso_psiquiatria.csv' (Dados do Gráfico: N e Taxas)")
    print("- 'base_estatistica_poliuso.csv' (Resultados do Teste de Correlação)")
    print(f"{'='*80}")

    # Variáveis para o gráfico a partir da tabela consolidada
    taxa_encaminhamento = tabela_export.set_index('Índice de Poliuso (Nº Drogas)')['Taxa de Encaminhamento (%)']
    contagem_casos = tabela_export.set_index('Índice de Poliuso (Nº Drogas)')['Total de Atendimentos (N)']

    # ==============================================================================
    # 6. VISUALIZAÇÃO PROFISSIONAL
    # ==============================================================================
    fig, ax1 = plt.subplots(figsize=(10, 6))
    sns.set_theme(style="whitegrid")

    # Gráfico de Barras: Taxa de Encaminhamento (%)
    sns.barplot(x=taxa_encaminhamento.index, y=taxa_encaminhamento.values, color='#3498db', ax=ax1)
    ax1.set_title('Impacto do Poliuso de Drogas no Encaminhamento Psiquiátrico', fontsize=14, pad=15, fontweight='bold')
    ax1.set_xlabel('Número de Substâncias Utilizadas (Índice de Poliuso)', fontsize=12)
    ax1.set_ylabel('Taxa de Encaminhamento para Psiquiatria (%)', fontsize=12, color='#2980b9')
    ax1.tick_params(axis='y', labelcolor='#2980b9')

    for p in ax1.patches:
        if p.get_height() > 0:
            ax1.annotate(f'{p.get_height():.1f}%',
                         (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='center', fontsize=11, fontweight='bold',
                         color='black', xytext=(0, 8), textcoords='offset points')

    # Eixo Secundário: Volume de Atendimentos
    ax2 = ax1.twinx()
    sns.lineplot(x=contagem_casos.index, y=contagem_casos.values, color='#e74c3c', marker='o', linewidth=2.5, ax=ax2)
    ax2.set_ylabel('Volume Total de Atendimentos (N)', fontsize=12, color='#c0392b')
    ax2.tick_params(axis='y', labelcolor='#c0392b')
    ax2.grid(False)

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # 7. RELATÓRIO ESTATÍSTICO (TERMINAL)
    # ==============================================================================
    print(f"\n{'='*80}")
    print("SÍNTESE ESTATÍSTICA: POLIUSUÁRIOS VS. PSIQUIATRIA")
    print(f"{'='*80}")
    print(tabela_export.to_string(index=False))
    print(f"\nTotal de Atendimentos Analisados: {len(df_ev)}\n")

    print("MÉTRICAS DO TESTE DE CORRELAÇÃO DE PONTO-BISERIAL:")
    print(f" -> Coeficiente de Correlação (r): {corr_coef:.4f}")
    print(f" -> Valor p (Significância):       {p_value:.4e}")
    print(f"{'='*80}")

    print("\nINTERPRETAÇÃO PARA GESTÃO:")
    if p_value < 0.05:
        direcao = "POSITIVA" if corr_coef > 0 else "NEGATIVA"
        print(f"Foi encontrada uma correlação ESTATISTICAMENTE SIGNIFICATIVA e {direcao} (p < 0.05).")
        if corr_coef > 0:
            print("Isso prova que, à medida que o paciente reporta o uso de múltiplas substâncias, a")
            print("probabilidade de o clínico geral encaminhá-lo para a Psiquiatria aumenta progressivamente.")
            print("Esse insight exige que a gestão dimensione adequadamente as vagas de Psiquiatria")
            print("no serviço de Telemedicina para absorver a demanda dessa população vulnerável.")
    else:
        print("NÃO foi encontrada correlação estatisticamente significativa entre o número de")
        print("drogas reportadas e o encaminhamento para a Psiquiatria nesta amostra.")
        print("O encaminhamento ocorre por outros fatores clínicos independentes do volume de dependência química.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CARREGAMENTO DA BASE DE DADOS
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: O arquivo {file_path} não foi encontrado.")
    raise

# ==============================================================================
# 2. IDENTIFICAÇÃO E UNIFICAÇÃO (TEXT MINING + ESTRUTURADO)
# ==============================================================================
text_cols = [c for c in df.columns if df[c].dtype == object]

# Mineração de texto linha a linha (Binário: 0 ou 1)
df['Uso_Crack'] = df[text_cols].apply(lambda x: x.astype(str).str.contains(r'\bcrack\b', case=False, na=False)).any(axis=1).astype(int)
df['Uso_Cocaina'] = df[text_cols].apply(lambda x: x.astype(str).str.contains(r'\bcoca[íi]na\b', case=False, na=False)).any(axis=1).astype(int)
df['Uso_Maconha'] = df[text_cols].apply(lambda x: x.astype(str).str.contains(r'\b(maconha|cannabis)\b', case=False, na=False)).any(axis=1).astype(int)
df['Uso_Heroina'] = df[text_cols].apply(lambda x: x.astype(str).str.contains(r'\bhero[íi]na\b', case=False, na=False)).any(axis=1).astype(int)
df['Uso_Opioide'] = df[text_cols].apply(lambda x: x.astype(str).str.contains(r'\bopi[óo]ide\b', case=False, na=False)).any(axis=1).astype(int)

# Captura de lícitas (Checkboxes)
cols_alcool = [c for c in df.columns if 'álcool' in c.lower() or 'alcool' in c.lower() and 'choice=' in c.lower()]
cols_tabaco = [c for c in df.columns if ('tabaco' in c.lower() or 'fumo' in c.lower() or 'cigarro' in c.lower()) and 'choice=' in c.lower()]

df['Uso_Alcool'] = df[cols_alcool].apply(lambda x: x.astype(str).str.contains('Checked|1|sim', case=False, na=False)).any(axis=1).astype(int) if cols_alcool else 0
df['Uso_Tabaco'] = df[cols_tabaco].apply(lambda x: x.astype(str).str.contains('Checked|1|sim', case=False, na=False)).any(axis=1).astype(int) if cols_tabaco else 0

# Psiquiatria (Multiplicamos por 100 para o gráfico já calcular a porcentagem direta)
cols_psiq = [c for c in df.columns if ('encaminhamento' in c.lower() or 'especialidade' in c.lower()) and ('psiquiatria' in c.lower() or 'saúde mental' in c.lower() or 'psicologia' in c.lower())]

if cols_psiq:
    df['Encaminhado_Psiq'] = df[cols_psiq].apply(lambda x: x.astype(str).str.contains('Checked|1|sim', case=False, na=False)).any(axis=1).astype(int) * 100
else:
    df['Encaminhado_Psiq'] = 0
    print("AVISO: Coluna de encaminhamento para Psiquiatria não encontrada.")

# ==============================================================================
# 3. A MÁGICA: AGREGAR POR PACIENTE ÚNICO (RECORD ID)
# ==============================================================================
cols_analise = ['Uso_Crack', 'Uso_Cocaina', 'Uso_Maconha', 'Uso_Heroina', 'Uso_Opioide', 'Uso_Alcool', 'Uso_Tabaco', 'Encaminhado_Psiq']

# Agrupamos pelo ID do paciente. Usamos .max() para que, se ele tiver "1" em qualquer consulta, ele fique como "1" no histórico geral.
df_user = df.groupby('Record ID')[cols_analise].max().reset_index()

# ==============================================================================
# 4. CLASSIFICAÇÃO DOS PERFIS CLÍNICOS E POLIUSO (POR PACIENTE)
# ==============================================================================
drogas_totais = ['Uso_Crack', 'Uso_Cocaina', 'Uso_Maconha', 'Uso_Heroina', 'Uso_Opioide', 'Uso_Alcool', 'Uso_Tabaco']
drogas_pesadas = ['Uso_Crack', 'Uso_Cocaina', 'Uso_Heroina', 'Uso_Opioide']

# Quantas drogas o paciente único usa na vida?
df_user['Total_Drogas_Usadas'] = df_user[drogas_totais].sum(axis=1)
df_user['Usou_Pesada'] = df_user[drogas_pesadas].sum(axis=1) > 0

def classificar_perfil(row):
    if row['Total_Drogas_Usadas'] == 0:
        return 'Não Usuário'
    if row['Usou_Pesada']:
        return 'Usuário de Drogas Pesadas'
    if row['Total_Drogas_Usadas'] > 1:
        return 'Poliusuário (Lícitas/Leves)'
    return 'Usuário Simples (1 Droga Leve)'

df_user['Perfil_Clinico'] = df_user.apply(classificar_perfil, axis=1)
df_user['Status_Crack'] = df_user['Uso_Crack'].apply(lambda x: 'Usuário de Crack' if x == 1 else 'Não usa Crack')

# ==============================================================================
# 5. LISTAGEM E EXPORTAÇÃO DAS BASES DE DADOS (CSVs)
# ==============================================================================
# Base 1: Ranking de Substâncias
ranking = df_user[drogas_totais].sum().sort_values(ascending=False)
ranking = ranking[ranking > 0]
ranking.index = [nome.replace('Uso_', '') for nome in ranking.index]

df_ranking = ranking.reset_index()
df_ranking.columns = ['Substância', 'Pacientes Únicos (N)']
df_ranking.to_csv('base_ranking_substancias.csv', index=False, encoding='utf-8-sig')

# Base 2: Perfil Clínico x Psiquiatria
tabela_perfil = df_user.groupby('Perfil_Clinico')['Encaminhado_Psiq'].agg(['mean', 'std', 'count']).reset_index()
tabela_perfil.columns = ['Perfil Clínico', 'Taxa de Encaminhamento Psiquiatria (%)', 'Desvio Padrão', 'N (Pacientes)']
ordem_perfil = ['Não Usuário', 'Usuário Simples (1 Droga Leve)', 'Poliusuário (Lícitas/Leves)', 'Usuário de Drogas Pesadas']
tabela_perfil['Perfil Clínico'] = pd.Categorical(tabela_perfil['Perfil Clínico'], categories=ordem_perfil, ordered=True)
tabela_perfil = tabela_perfil.sort_values('Perfil Clínico')
tabela_perfil.to_csv('base_perfil_drogas_psiquiatria.csv', index=False, encoding='utf-8-sig')

# Base 3: O Fator Crack x Psiquiatria
tabela_crack = df_user.groupby('Status_Crack')['Encaminhado_Psiq'].agg(['mean', 'std', 'count']).reset_index()
tabela_crack.columns = ['Status de Uso de Crack', 'Taxa de Encaminhamento Psiquiatria (%)', 'Desvio Padrão', 'N (Pacientes)']
ordem_crack = ['Não usa Crack', 'Usuário de Crack']
tabela_crack['Status de Uso de Crack'] = pd.Categorical(tabela_crack['Status de Uso de Crack'], categories=ordem_crack, ordered=True)
tabela_crack = tabela_crack.sort_values('Status de Uso de Crack')
tabela_crack.to_csv('base_fator_crack_psiquiatria.csv', index=False, encoding='utf-8-sig')

print(f"\n{'='*80}")
print(f"✅ BASES DE DADOS EXPORTADAS COM SUCESSO (N = {len(df_user)} Pacientes Únicos):")
print("- 'base_ranking_substancias.csv'")
print("- 'base_perfil_drogas_psiquiatria.csv'")
print("- 'base_fator_crack_psiquiatria.csv'")
print(f"{'='*80}")

print(f"\nRANKING DE SUBSTÂNCIAS:")
for droga, qtd in ranking.items():
    marcador = "[PESADA] " if droga in ['Crack', 'Cocaina', 'Heroina', 'Opioide'] else ""
    print(f" -> {marcador}{droga:<30} : {int(qtd)} pacientes únicos")

# ==============================================================================
# 6. VISUALIZAÇÃO: PAINEL TRIPLO DE PATOLOGIA DUAL POR USUÁRIO
# ==============================================================================
sns.set_theme(style="whitegrid")
fig = plt.figure(figsize=(18, 12))

ax1 = plt.subplot2grid((2, 2), (0, 0), colspan=2)
ax2 = plt.subplot2grid((2, 2), (1, 0))
ax3 = plt.subplot2grid((2, 2), (1, 1))

# --- GRÁFICO 1: RANKING DE DROGAS ---
cores_ranking = ['#c0392b' if d in ['Crack', 'Cocaina', 'Heroina', 'Opioide'] else '#7f8c8d' for d in ranking.index]

sns.barplot(x=ranking.values, y=ranking.index, palette=cores_ranking, ax=ax1)
ax1.set_title("Ranking de Substâncias Mais Utilizadas (Pacientes ÚNICOS)", fontsize=15, fontweight='bold', pad=15)
ax1.set_xlabel("Número de Pacientes Únicos", fontsize=12, fontweight='bold')
ax1.set_ylabel("")

if cols_psiq:
    # --- GRÁFICO 2: O IMPACTO DO POLIUSO E DROGAS PESADAS ---
    sns.barplot(data=df_user, x='Perfil_Clinico', y='Encaminhado_Psiq', order=ordem_perfil,
                palette=['#2ecc71', '#f1c40f', '#e67e22', '#c0392b'], capsize=.05, err_kws={'linewidth': 1.5}, ax=ax2)

    ax2.set_title("O Efeito Cascata: Poliuso e Drogas Pesadas na Psiquiatria", fontsize=14, fontweight='bold', pad=15)
    ax2.set_ylabel("% de Pacientes Encaminhados à Psiquiatria", fontsize=12, fontweight='bold')
    ax2.set_xlabel("")
    ax2.tick_params(axis='x', rotation=15)

    # --- GRÁFICO 3: O FATOR CRACK ---
    sns.barplot(data=df_user, x='Status_Crack', y='Encaminhado_Psiq', order=ordem_crack,
                palette=['#95a5a6', '#8e44ad'], capsize=.1, err_kws={'linewidth': 1.5}, ax=ax3)

    for i, p in enumerate(ax3.patches):
        ax3.annotate(f"{p.get_height():.1f}%",
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center', xytext=(0, 10), textcoords='offset points', fontweight='bold')

    ax3.set_title("O Fator Crack (Patologia Dual)", fontsize=14, fontweight='bold', pad=15)
    ax3.set_ylabel("% de Pacientes Encaminhados à Psiquiatria", fontsize=12, fontweight='bold')
    ax3.set_xlabel("")

sns.despine(bottom=True, left=True)
plt.tight_layout(pad=3.0)
plt.show()

# ==============================================================================
# 7. RELATÓRIO ESTATÍSTICO DE CHANCE POR PACIENTE (ODDS)
# ==============================================================================
if cols_psiq:
    print(f"\n{'='*90}")
    print("SÍNTESE CLÍNICA: RISCO DE ENCAMINHAMENTO PSIQUIÁTRICO (POR PACIENTE ÚNICO)")
    print(f"{'='*90}")

    media_nao_uso = df_user[df_user['Perfil_Clinico'] == 'Não Usuário']['Encaminhado_Psiq'].mean()
    media_poliuso = df_user[df_user['Perfil_Clinico'] == 'Poliusuário (Lícitas/Leves)']['Encaminhado_Psiq'].mean()
    media_pesadas = df_user[df_user['Perfil_Clinico'] == 'Usuário de Drogas Pesadas']['Encaminhado_Psiq'].mean()

    if media_nao_uso > 0:
        print(f" -> POLIUSUÁRIOS de drogas leves têm {media_poliuso/media_nao_uso:.1f}x mais chance de necessitar de Psiquiatria.")
        print(f" -> USUÁRIOS DE DROGAS PESADAS (Cocaína/Crack/Opioides) têm {media_pesadas/media_nao_uso:.1f}x mais chance de necessitar de Psiquiatria.")

    media_sem_crack = df_user[df_user['Status_Crack'] == 'Não usa Crack']['Encaminhado_Psiq'].mean()
    media_com_crack = df_user[df_user['Status_Crack'] == 'Usuário de Crack']['Encaminhado_Psiq'].mean()

    if media_sem_crack > 0:
        print(f"\n[FOCO NO CRACK]")
        print(f" -> Taxa Psiquiátrica entre os NÃO usuários de Crack: {media_sem_crack:.1f}%")
        print(f" -> Taxa Psiquiátrica entre os USUÁRIOS de Crack: {media_com_crack:.1f}%")
        print(f" -> Um usuário de CRACK possui {media_com_crack/media_sem_crack:.1f}x mais chance de ser encaminhado para saúde mental na sua base.")
    print(f"{'='*90}\n")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CARREGAMENTO DOS DADOS
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: O arquivo {file_path} não foi encontrado.")
    raise

# ==============================================================================
# 2. IDENTIFICAÇÃO DAS COLUNAS (PSIQUIATRIA E CIAP)
# ==============================================================================
cols_esp = [c for c in df.columns if ('encaminhamento' in c.lower() or 'especialidade' in c.lower()) and 'choice=' in c.lower()]
cols_ciap = [c for c in df.columns if 'ciap' in c.lower()]

# Procura especificamente a coluna da Psiquiatria
col_psiq = next((c for c in cols_esp if 'psiquiatria' in c.lower() or 'saúde mental' in c.lower() or 'psicologia' in c.lower()), None)

if not col_psiq:
    print("ERRO: Coluna de Psiquiatria/Saúde Mental não encontrada.")
elif not cols_ciap:
    print("ERRO: Coluna de Motivo da Consulta (CIAP-2) não encontrada.")
else:
    col_ciap_alvo = cols_ciap[0]

    # ==============================================================================
    # 3. FILTRAGEM: ISOLANDO A FILA DA PSIQUIATRIA
    # ==============================================================================
    # Transforma os marcadores em booleanos e filtra o DataFrame
    is_psiq = df[col_psiq].astype(str).str.lower().str.strip().isin(['checked', '1', '1.0', 'sim', 'verdadeiro'])
    df_psiq = df[is_psiq].copy()

    total_guias_psiq = len(df_psiq)

    if total_guias_psiq == 0:
        print("Nenhum encaminhamento para Psiquiatria foi encontrado com esses critérios.")
    else:
        # ==============================================================================
        # 4. GERAÇÃO E EXPORTAÇÃO DA BASE DE DADOS (NOVO!)
        # ==============================================================================
        # Conta a frequência de todos os motivos CIAP-2 na fila da Psiquiatria
        tabela_motivos_psiq = df_psiq[col_ciap_alvo].dropna().value_counts().reset_index()
        tabela_motivos_psiq.columns = ['Motivo da Consulta (CIAP-2)', 'Frequência Absoluta (N)']
        tabela_motivos_psiq['Proporção (%)'] = (tabela_motivos_psiq['Frequência Absoluta (N)'] / total_guias_psiq * 100).round(2)
        
        # Exporta a base completa para CSV
        nome_arquivo_export = 'base_motivos_psiquiatria.csv'
        tabela_motivos_psiq.to_csv(nome_arquivo_export, index=False, encoding='utf-8-sig')

        print(f"\n{'='*80}")
        print("✅ BASE DE DADOS EXPORTADA COM SUCESSO:")
        print(f"- '{nome_arquivo_export}' (Fila completa da Psiquiatria com N e Proporção)")
        print(f"{'='*80}")

        # ==============================================================================
        # 5. CONTAGEM DOS MOTIVOS PARA O GRÁFICO (TOP 7)
        # ==============================================================================
        # Pega os 7 motivos mais frequentes para não poluir o gráfico
        top_motivos = tabela_motivos_psiq.head(7).copy()

        # Encurta os nomes do CIAP se forem muito longos (para o gráfico não ficar espremido)
        top_motivos['Nome Curto'] = top_motivos['Motivo da Consulta (CIAP-2)'].apply(
            lambda m: str(m)[:65] + ('...' if len(str(m)) > 65 else '')
        )
        
        # Configura as séries para o gráfico
        valores_grafico = top_motivos['Frequência Absoluta (N)'].values
        labels_grafico = top_motivos['Nome Curto'].values

        # ==============================================================================
        # 6. MÁGICA VISUAL: O RAIO-X DA PSIQUIATRIA
        # ==============================================================================
        sns.set_theme(style="whitegrid")
        plt.figure(figsize=(14, 7))

        # Destaca o principal motivo em vermelho vibrante, os demais em cinza
        paleta_motivos = ['#e74c3c'] + ['#bdc3c7'] * (len(top_motivos) - 1)

        ax = sns.barplot(x=valores_grafico, y=labels_grafico, palette=paleta_motivos)

        plt.title(f"Raio-X da Psiquiatria: Por que os pacientes são encaminhados?\n(Base de análise: {total_guias_psiq} guias emitidas)",
                  fontsize=16, fontweight='bold', pad=20)
        plt.xlabel("Volume Absoluto de Casos", fontsize=13, fontweight='bold')
        plt.ylabel("Motivo da Consulta (Código CIAP-2)", fontsize=13, fontweight='bold')

        # Adiciona os números e porcentagens diretamente nas barras
        for p in ax.patches:
            qtd = int(p.get_width())
            perc = (qtd / total_guias_psiq) * 100
            ax.annotate(f"  {qtd} casos ({perc:.1f}%)",
                        (qtd, p.get_y() + p.get_height() / 2.),
                        ha='left', va='center', xytext=(5, 0), textcoords='offset points',
                        fontweight='bold', fontsize=11, color='#2c3e50')

        sns.despine(left=True, bottom=True)
        # Expande o eixo X em 20% para a porcentagem não cortar na margem direita
        plt.xlim(0, max(valores_grafico) * 1.20)
        plt.tight_layout()
        plt.show()

        # ==============================================================================
        # 7. SÍNTESE CLÍNICA (CONSOLE)
        # ==============================================================================
        print(f"\n{'='*90}")
        print(f"DIAGNÓSTICO DA FILA DE PSIQUIATRIA (N = {total_guias_psiq})")
        print(f"{'='*90}")

        motivo_campeao = tabela_motivos_psiq.iloc[0]['Motivo da Consulta (CIAP-2)']
        porcentagem_campeao = tabela_motivos_psiq.iloc[0]['Proporção (%)']

        print(f"O principal ofensor da fila psiquiátrica na TeleSAP é:\n-> '{motivo_campeao}'\n")
        print(f"Esse único motivo é responsável por {porcentagem_campeao:.1f}% de TODOS os encaminhamentos para a especialidade.")

        if 'ansiedade' in motivo_campeao.lower() or 'depressivo' in motivo_campeao.lower():
            print("\n[ALERTA DE RESOLUTIVIDADE]")
            print("Como o motivo principal trata-se de um Transtorno Mental Comum (Ansiedade/Depressão Leve a Moderada), ")
            print("há um alto potencial de resolução na própria Atenção Primária, desde que os médicos recebam capacitação ")
            print("e protocolos claros para prescrição de ISRS (antidepressivos) e manejo clínico inicial.")
        print(f"{'='*90}\n")